# Week 9, Day 2: Linearisation and the Jacobian: local linear maps

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w009/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A numerical Jacobian

Write `numerical_jacobian(F, x, h=1e-6)`. Build it column by column: column j is the central difference of the whole output vector when input j is nudged. Test it on the polar-to-Cartesian map, whose Jacobian determinant is r.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def numerical_jacobian(F, x, h=1e-6):
    m = len(F(x))
    J = np.zeros((m, len(x)))
    for j in range(len(x)):
        e = np.zeros(len(x)); e[j] = h
        J[:, j] = ...
    return J

F = lambda z: np.array([z[0] * np.cos(z[1]), z[0] * np.sin(z[1])])
Jp = numerical_jacobian(F, np.array([2.0, 0.7]))
print(Jp.round(4), " det =", round(np.linalg.det(Jp), 4))

In [ ]:
import numpy as np
def test_linear():
    "the Jacobian of x -> A x is A"
    A = np.array([[1.0, 2.0, 0.0], [0.0, -1.0, 3.0]])
    assert np.allclose(numerical_jacobian(lambda x: A @ x, np.ones(3)), A, atol=1e-6)

def test_polar():
    "the Jacobian of (r cos t, r sin t) matches the analytic one"
    F = lambda z: np.array([z[0] * np.cos(z[1]), z[0] * np.sin(z[1])])
    r, t = 2.0, 0.7
    J = np.array([[np.cos(t), -r * np.sin(t)], [np.sin(t), r * np.cos(t)]])
    assert np.allclose(numerical_jacobian(F, np.array([r, t])), J, atol=1e-6)

test_linear()
test_polar()
print("All checks passed")

### Exercise: Jacobians through time

Write `step_jacobian(h)` for the network below. Then run 10 steps from `h0`, multiplying the step Jacobians in the right order (each new one on the left) to get `J_total`, the Jacobian of the whole 10-step map. Compare its largest singular value with that of a single step.

In [ ]:
rng = np.random.default_rng(9)
n = 6
W = rng.normal(size=(n, n)) * 1.5 / np.sqrt(n)
b = rng.normal(size=n) * 0.3
def step(h):
    return np.tanh(W @ h + b)

def step_jacobian(h):
    hp = step(h)
    return ...

h0 = rng.normal(size=n) * 0.5
h, J_total = h0.copy(), np.eye(n)
for _ in range(10):
    J_total = ...
    h = step(h)

print("largest singular value: one step", np.linalg.svd(step_jacobian(h0), compute_uv=False)[0].round(3),
      "  ten steps", np.linalg.svd(J_total, compute_uv=False)[0].round(4))

In [ ]:
import numpy as np
def test_step():
    "step_jacobian matches the numerical Jacobian of one step"
    h = np.random.default_rng(3).normal(size=n) * 0.5
    assert np.allclose(step_jacobian(h), numerical_jacobian(step, h), atol=1e-6)

def test_product():
    "J_total, the product of step Jacobians along the trajectory, matches the numerical Jacobian of 10 steps"
    def ten(h):
        for _ in range(10):
            h = step(h)
        return h
    assert np.allclose(J_total, numerical_jacobian(ten, h0), atol=1e-5)

test_step()
test_product()
print("All checks passed")

### Exercise: Predict recovery from the Jacobian

Find the fixed point `x_star` by simulating the network for a long time. Compute `A` with the formula above, and the predicted time constant `tau_pred` of its slowest mode. Then kick the network slightly, simulate its recovery, and measure the time constant `tau_measured` from the slope of the log distance to the fixed point between 50 and 150 ms.

In [ ]:
W2 = np.array([[0.9, -0.6], [0.5, 0.4]])
b2 = np.array([0.3, -0.1])
tau, dt = 0.01, 0.0001                      # 10 ms, 0.1 ms steps
def simulate(x, n_steps):
    xs = [x.copy()]
    for _ in range(n_steps):
        x = x + dt / tau * (-x + np.tanh(W2 @ x + b2))
        xs.append(x.copy())
    return np.array(xs)

x_star = simulate(np.zeros(2), 20000)[-1]
A = ...
lam = np.linalg.eigvals(A)
tau_pred = ...                              # for the slowest mode

xs = simulate(x_star + np.array([0.05, -0.03]), 2000)
t = np.arange(2001) * dt
dist = np.linalg.norm(xs - x_star, axis=1)
k = (t >= 0.05) & (t <= 0.15)
tau_measured = ...

plt.figure(figsize=(5, 3)); plt.semilogy(t * 1000, dist); plt.xlabel("time (ms)"); plt.ylabel("distance from fixed point"); plt.show()
print("eigenvalues of A:", lam.round(3), f"  predicted {1000 * tau_pred:.2f} ms, measured {1000 * tau_measured:.2f} ms")

In [ ]:
import numpy as np
def test_fixed():
    "x_star is a fixed point"
    assert np.allclose(-x_star + np.tanh(W2 @ x_star + b2), 0, atol=1e-8)

def test_A():
    "A is the Jacobian of the right-hand side at the fixed point"
    rhs = lambda x: -x + np.tanh(W2 @ x + b2)
    assert np.allclose(A, numerical_jacobian(rhs, x_star), atol=1e-6)

def test_prediction():
    "the measured decay time constant matches tau / |Re lambda| of the slowest mode within 5%"
    assert abs(tau_measured - tau_pred) / tau_pred < 0.05

test_fixed()
test_A()
test_prediction()
print("All checks passed")